# Time-series features, revisited: do they predict anything?

**Time-series features · versions 2 (Part 1) and 3 (Part 2) · 11 November 2026.** Version 1 is `time_series.ipynb` (git tag `v1`), unchanged. History: `VERSIONS.md`.

The original notebook (`time_series.ipynb`, tagged `v1`) builds and charts seven features of AAPL's price and hands them to the trading agent. It describes what the price has done. It never tests whether any feature says something about what the price does next. This notebook runs that test and leaves the original untouched.

| Section | Question |
|---|---|
| 1 | How much independent information is in the seven features? |
| 2 | On AAPL, does any feature predict the next 1, 5 or 21 days' return? |
| 3 | How small an effect could 1,100 days even detect? |
| 4 | Do the features predict the next month's *volatility* instead? |
| 5 | With 300 to 400 stocks instead of one: does any feature rank stocks within their sector? |

**Data.** The feature file the original notebook saved (AAPL, March 2020 to August 2024), and for section 5 a frozen S&P 500 snapshot that ends in December 2021 (later data is held back for another project's final test). Nothing here is traded, so there are no costs; an effect found here would still have to survive them.

In [1]:
import json
import os
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go

for path in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), '..'))):
    if os.path.exists(os.path.join(path, 'func_lib.py')) and path not in sys.path:
        sys.path.append(path)
import cross_section as cs
import func_lib
import ts_features as tf

TICKER = 'AAPL'
ORIGINAL_SEVEN = ['Close', 'SMA_20', 'SMA_50', 'Volatility', 'BB_upper', 'BB_lower', 'RSI']
HORIZONS = [1, 5, 21]

## 1. Seven features, how much information?

In [2]:
data = pd.read_csv(f'{TICKER}_time_series_features.csv', index_col=0, parse_dates=True)
features = tf.stationary_features(data)                     # the model inputs, re-expressed as scale-free ratios
raw = data[ORIGINAL_SEVEN]
print(f'{TICKER}: {len(data):,} days, {data.index[0]:%Y-%m-%d} to {data.index[-1]:%Y-%m-%d}\n')

info = pd.DataFrame({'original seven (levels)': tf.effective_count(raw),
                     'original seven (daily changes, as the agent reads them)': tf.effective_count(raw.diff()),
                     'scale-free versions': tf.effective_count(features)}).T
print(info.round(2).to_string())
print('\nCorrelation of the original seven (levels):')
print(raw.corr().round(2).to_string())
corr = features.corr()
fig = go.Figure(go.Heatmap(z=corr.to_numpy(), x=corr.columns, y=corr.index, zmin=-1, zmax=1, colorscale='RdBu', reversescale=True,
                           text=corr.round(2).to_numpy(), texttemplate='%{text}'))
fig.update_layout(title='Correlation between the scale-free features', height=460)
fig.show()

AAPL: 1,124 days, 2020-03-13 to 2024-08-29

                                                         share_first  effective  features
original seven (levels)                                         0.72       1.81       7.0
original seven (daily changes, as the agent reads them)         0.35       3.41       7.0
scale-free versions                                             0.57       2.49       7.0

Correlation of the original seven (levels):
            Close  SMA_20  SMA_50  Volatility  BB_upper  BB_lower   RSI
Close        1.00    0.98    0.96        0.28      0.98      0.98  0.08
SMA_20       0.98    1.00    0.98        0.27      1.00      0.99 -0.09
SMA_50       0.96    0.98    1.00        0.25      0.98      0.98 -0.18
Volatility   0.28    0.27    0.25        1.00      0.36      0.16  0.01
BB_upper     0.98    1.00    0.98        0.36      1.00      0.98 -0.08
BB_lower     0.98    0.99    0.98        0.16      0.98      1.00 -0.09
RSI          0.08   -0.09   -0.18        0.01  

`share_first` is the share of all variation carried by one combination of the features; `effective` is how many independent features they are worth. Five of the original seven are price levels in dollars, so in levels they are close to one number repeated. The scale-free versions ask the same questions (how far above the average, where inside the band, how wide the band) in units that mean the same at $67 and at $228.

## 2. Does any feature predict AAPL's return?

For each feature and horizon: the rank correlation between the feature today and the return over the following days, with a t-statistic that allows for overlapping horizons. An absolute t below 2 is indistinguishable from no relationship.

In [3]:
close = data['Close']
ret_tables = {}
for h in HORIZONS:
    fwd = close.shift(-h) / close - 1
    ret_tables[h] = tf.predictive_table(features, fwd, h)
ret_view = pd.concat({f'{h}d': t[['corr', 't']] for h, t in ret_tables.items()}, axis=1)
print('Feature today vs AAPL return over the next h days:')
print(ret_view.round(2).to_string())
n_tests = len(features.columns) * len(HORIZONS)
n_sig = int(sum((t['t'].abs() > 2).sum() for t in ret_tables.values()))
print(f'\n{n_sig} of {n_tests} tests have |t| > 2; about {0.05 * n_tests:.1f} would by chance alone.')

Feature today vs AAPL return over the next h days:
              1d          5d         21d      
            corr     t  corr     t  corr     t
dist_sma20  0.01  0.21  0.05  1.00  0.03  0.37
dist_sma50 -0.00 -0.11  0.00  0.07 -0.04 -0.37
sma_spread -0.00 -0.06 -0.03 -0.57 -0.11 -0.94
band_pos    0.01  0.34  0.05  0.99  0.05  0.68
band_width  0.02  0.79  0.05  0.88 -0.03 -0.26
rsi         0.00  0.06  0.03  0.59  0.01  0.09
ret_vol     0.01  0.26  0.00  0.05  0.02  0.15

0 of 21 tests have |t| > 2; about 1.1 would by chance alone.


## 3. What could this sample detect at all?

In [4]:
n = len(data)
se_1d = 1 / np.sqrt(n)
print(f'With {n:,} daily observations, a correlation has a standard error of about {se_1d:.3f}.')
print(f'So only a correlation beyond about {2 * se_1d:.2f} would stand out at a 1-day horizon.')
for h in (5, 21):
    print(f'At {h} days the outcomes overlap, leaving about {n // h} independent periods: detectable beyond about {2 / np.sqrt(n / h):.2f}.')
print('Published return predictors of this kind have correlations of roughly 0.02 to 0.05 with next-period returns.')

With 1,124 daily observations, a correlation has a standard error of about 0.030.
So only a correlation beyond about 0.06 would stand out at a 1-day horizon.
At 5 days the outcomes overlap, leaving about 224 independent periods: detectable beyond about 0.13.
At 21 days the outcomes overlap, leaving about 53 independent periods: detectable beyond about 0.27.
Published return predictors of this kind have correlations of roughly 0.02 to 0.05 with next-period returns.


## 4. Do the features predict volatility instead?

Same test, different outcome: the realized volatility of the next 21 days.

In [5]:
H = 21
fwd_vol = (data['Log_Return'] ** 2).rolling(H).mean().shift(-H).pipe(np.sqrt) * np.sqrt(252)
vol_table = tf.predictive_table(features, fwd_vol, H)
print('Feature today vs realized volatility of the next 21 days:')
print(vol_table[['corr', 't']].round(2).to_string())

Feature today vs realized volatility of the next 21 days:
            corr     t
dist_sma20 -0.10 -1.08
dist_sma50 -0.10 -0.81
sma_spread -0.01 -0.04
band_pos   -0.17 -1.82
band_width  0.48  6.67
rsi        -0.14 -1.22
ret_vol     0.62  8.81


## 5. One stock is one history: the same test across the S&P 500

One stock gives about 1,100 days. Several hundred stocks give several hundred times as many observations of the same question, asked in the form a trader would use: *within its sector, does a stock that is high on this feature do better over the next 21 days than one that is low?* The **information coefficient (IC)** is that daily rank correlation. Each company counts only from the date it joined the index. Two halves are shown: a relationship that is real should have the same sign in both.

In [6]:
panel = cs.load_sp500_panel(point_in_time=True)
wide = tf.wide_features(panel.close)
fwd21 = cs.forward_return(panel.close, H)
PERIODS = {'2010-2015': ('2010-01-01', '2015-12-31'), '2016-2021': ('2016-01-01', '2021-12-31')}
rows = {}
for k, f in wide.items():
    ic = cs.daily_rank_ic(f, fwd21, panel.sectors, panel.member)
    rows[k] = {}
    for name, (lo, hi) in PERIODS.items():
        s = cs.ic_summary(ic.loc[lo:hi], H)
        rows[k][f'IC {name}'], rows[k][f't {name}'] = s['IC'], s['t']
    s = cs.ic_summary(ic.loc['2010-01-01':'2021-12-31'], H)
    rows[k]['IC all'], rows[k]['t all'] = s['IC'], s['t']
ic_table = pd.DataFrame(rows).T
ic_table['same sign'] = np.sign(ic_table['IC 2010-2015']) == np.sign(ic_table['IC 2016-2021'])
print(f"Average stocks per day: {int(panel.member.loc['2010':'2021'].sum(axis=1).mean())}")
print(ic_table.round(3).to_string())

Average stocks per day: 338
            IC 2010-2015  t 2010-2015  IC 2016-2021  t 2016-2021  IC all  t all  same sign
dist_sma20        -0.005       -0.379        -0.026       -1.524  -0.015 -1.454       True
dist_sma50        -0.000       -0.012        -0.021       -1.176  -0.010 -0.933       True
sma_spread         0.002        0.155        -0.010       -0.619  -0.004 -0.375      False
band_pos          -0.006       -0.508        -0.021       -1.439  -0.013 -1.447       True
band_width        -0.020       -1.319         0.013        0.799  -0.003 -0.294      False
rsi               -0.002       -0.183        -0.022       -1.280  -0.012 -1.126       True
ret_vol           -0.010       -0.507         0.019        0.918   0.004  0.301      False


## 6. Findings

In [7]:
# Written from the numbers above, and saved for the website
best_ret = max(((h, k, ret_tables[h].loc[k, 't']) for h in HORIZONS for k in features.columns), key=lambda x: abs(x[2]))
vol_sig = vol_table.index[vol_table['t'].abs() > 2].tolist()
top_vol = vol_table['corr'].abs().idxmax()
stable = ic_table[(ic_table['same sign']) & (ic_table['t all'].abs() > 2)]
top_ic = ic_table['IC all'].abs().idxmax()
print(f"1. Information: in levels, one combination carries {100 * info.loc['original seven (levels)', 'share_first']:.0f}% of the variation of the original seven "
      f"(worth {info.loc['original seven (levels)', 'effective']:.1f} independent features). The scale-free versions are worth {info.loc['scale-free versions', 'effective']:.1f}.")
print(f"2. AAPL returns: {n_sig} of {n_tests} tests reach |t| > 2 (chance alone gives about {0.05 * n_tests:.1f}). "
      f"The strongest is {best_ret[1]} at {best_ret[0]} days (t {best_ret[2]:.1f}).")
print(f"3. Power: this sample can only detect correlations beyond about {2 * se_1d:.2f} at 1 day and {2 / np.sqrt(n / 21):.2f} at 21 days; known effects are 0.02 to 0.05.")
print(f"4. Volatility: {len(vol_sig)} of {len(vol_table)} features predict next-month volatility with |t| > 2; the strongest is {top_vol} "
      f"(correlation {vol_table.loc[top_vol, 'corr']:.2f}, t {vol_table.loc[top_vol, 't']:.1f}).")
print(f"5. Across the S&P 500: the largest IC is {top_ic} ({ic_table.loc[top_ic, 'IC all']:+.3f}, t {ic_table.loc[top_ic, 't all']:.1f}). "
      f"Features with |t| > 2 overall AND the same sign in both halves: {', '.join(stable.index) if len(stable) else 'none'}.")

summary = {'ticker': TICKER, 'days': int(n), 'first': str(data.index[0].date()), 'last': str(data.index[-1].date()),
           'info': json.loads(info.to_json(orient='index')),
           'returns': {str(h): json.loads(t.to_json(orient='index')) for h, t in ret_tables.items()}, 'n_tests': n_tests, 'n_sig': n_sig,
           'detectable': {'1': float(2 * se_1d), '21': float(2 / np.sqrt(n / 21))},
           'vol': json.loads(vol_table.to_json(orient='index')),
           'ic': json.loads(ic_table.astype(float).to_json(orient='index')), 'stocks_per_day': int(panel.member.loc['2010':'2021'].sum(axis=1).mean())}
with open(os.path.join('data_cache', 'time_series_revisited_summary.json'), 'w') as f:
    json.dump(summary, f, indent=1)

1. Information: in levels, one combination carries 72% of the variation of the original seven (worth 1.8 independent features). The scale-free versions are worth 2.5.
2. AAPL returns: 0 of 21 tests reach |t| > 2 (chance alone gives about 1.1). The strongest is dist_sma20 at 5 days (t 1.0).
3. Power: this sample can only detect correlations beyond about 0.06 at 1 day and 0.27 at 21 days; known effects are 0.02 to 0.05.
4. Volatility: 2 of 7 features predict next-month volatility with |t| > 2; the strongest is ret_vol (correlation 0.62, t 8.8).
5. Across the S&P 500: the largest IC is dist_sma20 (-0.015, t -1.5). Features with |t| > 2 overall AND the same sign in both halves: none.


---
# Part 2: eight new features

Part 1 found that the original seven describe risk, not direction. This part tries eight features chosen to carry different information, and asks the same two questions of each: does it predict the next 21 days' **return**, and the next 21 days' **volatility**?

| Group | Feature | What it measures |
|---|---|---|
| Risk state | `gk_vol` | volatility from each day's full range (open, high, low, close), not only the close |
| | `downside_vol` | volatility counting only down days |
| | `vol_of_vol` | how unstable volatility itself has been |
| Relative | `rel_ret_21` | the stock's 21-day return minus its sector's |
| Events | `sue` | the latest earnings surprise, dated by the **announcement** (not the later quarterly filing) |
| | `days_since_earnings` | trading days since that announcement |
| Market state | `vix_pct` | where VIX stands in its own history so far |
| | `vix_slope` | 3-month VIX over 1-month VIX, minus 1: positive in calm markets, negative in stress |

**Pass mark, written to `data_cache/features_round2_prereg.json` before any of this was run.** A feature passes for an outcome only if its t-statistic exceeds 2 over 2010 to 2021, its sign is the same in 2010–2015 and 2016–2021, and it survives the Benjamini–Hochberg correction at 10% across the eight.

**Two kinds of test.** The six stock-specific features are tested the way Part 1 section 5 did: within its sector, does a stock that is high on the feature do better (or turn out more volatile) than one that is low? The two market-state features are the same for every stock on a given day, so they cannot rank stocks; they are tested through time against the average stock's outcome.

**Data.** Frozen snapshots ending December 2021: daily open, high, low, close; VIX and 3-month VIX; and earnings announcements taken from SEC filings (each company files its earnings release as a Form 8-K, Item 2.02; the surprise is this quarter's net income minus the same quarter a year earlier, scaled by how variable that change has been).

In [8]:
ROUND2 = ('2010-01-01', '2021-12-31')
HALVES = {'2010-2015': ('2010-01-01', '2015-12-31'), '2016-2021': ('2016-01-01', '2021-12-31')}
read = lambda f: pd.read_csv(os.path.join('data_cache', f'sp500_ohlcv_{f}_2006_2021.csv.gz'), index_col=0, parse_dates=True).reindex(
    index=panel.close.index, columns=panel.close.columns)
o, hi, lo, c = read('open'), read('high'), read('low'), read('close')
vix = pd.read_csv(os.path.join('data_cache', 'vix_term_2006_2021.csv'), index_col=0, parse_dates=True).reindex(panel.close.index).ffill()
events = pd.read_csv(os.path.join('data_cache', 'sp500_earnings_events_to_2021.csv'), parse_dates=['quarter_end', 'announced', 'filing_date'])

stock_feats = {**tf.risk_features(o, hi, lo, c), 'rel_ret_21': tf.relative_return(c, panel.sectors, panel.member),
               **tf.event_features(events, panel.close.index, panel.close.columns)}
state = tf.market_state(vix['^VIX'], vix['^VIX3M'])
fwd_ret = cs.forward_return(panel.close, H)
log_r = np.log(panel.close).diff()
fwd_vol = np.sqrt((log_r ** 2).rolling(H).mean().shift(-H) * 252)

lead = (events['filing_date'] - events['announced']).dt.days[events['source'] == '8-K']
print(f"{len(events):,} earnings surprises for {events['ticker'].nunique()} companies; {100 * (events['source'] == '8-K').mean():.0f}% dated by the announcement, "
      f"a median of {lead.median():.0f} days before the quarterly filing.")
cover = pd.Series({k: float((v.where(panel.member).loc[ROUND2[0]:ROUND2[1]].notna().sum(axis=1) / panel.member.loc[ROUND2[0]:ROUND2[1]].sum(axis=1)).mean())
                   for k, v in stock_feats.items()})
print('Share of index members with each stock-specific feature, 2010-2021:', (100 * cover).round(0).astype(int).to_dict())

19,320 earnings surprises for 476 companies; 99% dated by the announcement, a median of 6 days before the quarterly filing.
Share of index members with each stock-specific feature, 2010-2021: {'gk_vol': 100, 'downside_vol': 100, 'vol_of_vol': 100, 'rel_ret_21': 100, 'sue': 88, 'days_since_earnings': 88}


## 8. The eight features against the pass mark

In [9]:
from scipy.stats import norm


def cross_section_test(feature, outcome):
    ic = cs.daily_rank_ic(feature, outcome, panel.sectors, panel.member)
    out = {'effect': cs.ic_summary(ic.loc[ROUND2[0]:ROUND2[1]], H)['IC'], 't': cs.ic_summary(ic.loc[ROUND2[0]:ROUND2[1]], H)['t']}
    for name, (a, b) in HALVES.items():
        out[name] = cs.ic_summary(ic.loc[a:b], H)['IC']
    return out


def time_series_test(feature, outcome):
    one = lambda a, b: tf.predictive_table(feature.loc[a:b].to_frame('x'), outcome.loc[a:b], H).loc['x']
    full = one(*ROUND2)
    out = {'effect': full['corr'], 't': full['t']}
    for name, (a, b) in HALVES.items():
        out[name] = one(a, b)['corr']
    return out


market_ret = fwd_ret.where(panel.member).mean(axis=1)            # the average member's next-21-day return
market_vol = fwd_vol.where(panel.member).median(axis=1)          # the typical member's next-21-day volatility
verdicts = {}
for label, outcome_wide, outcome_mkt in (('return', fwd_ret, market_ret), ('volatility', fwd_vol, market_vol)):
    rows = {k: {'test': 'ranks stocks within sector', **cross_section_test(v, outcome_wide)} for k, v in stock_feats.items()}
    rows.update({k: {'test': 'through time, market average', **time_series_test(state[k], outcome_mkt)} for k in state})
    t = pd.DataFrame(rows).T
    t['same sign'] = np.sign(t['2010-2015'].astype(float)) == np.sign(t['2016-2021'].astype(float))
    t['p'] = 2 * norm.sf(t['t'].astype(float).abs())
    t['survives BH'] = tf.benjamini_hochberg(t['p'].astype(float))
    t['PASS'] = (t['t'].astype(float).abs() > 2) & t['same sign'] & t['survives BH']
    verdicts[label] = t
    print(f"\nOutcome: next-21-day {label}   (effect = IC for stock rankings, rank correlation for market-state features)")
    show = t[['test', 'effect', 't', '2010-2015', '2016-2021', 'same sign', 'p', 'survives BH', 'PASS']].copy()
    for col in ('effect', 't', '2010-2015', '2016-2021', 'p'):
        show[col] = show[col].astype(float).round(3)
    print(show.to_string())


Outcome: next-21-day return   (effect = IC for stock rankings, rank correlation for market-state features)
                                             test  effect      t  2010-2015  2016-2021  same sign      p  survives BH   PASS
gk_vol                 ranks stocks within sector   0.006  0.390     -0.005      0.017      False  0.696        False  False
downside_vol           ranks stocks within sector   0.008  0.538     -0.011      0.027      False  0.590        False  False
vol_of_vol             ranks stocks within sector  -0.001 -0.147     -0.002      0.000      False  0.883        False  False
rel_ret_21             ranks stocks within sector  -0.013 -1.189     -0.002     -0.023       True  0.235        False  False
sue                    ranks stocks within sector   0.006  0.820      0.015     -0.001      False  0.412        False  False
days_since_earnings    ranks stocks within sector   0.003  0.424      0.003      0.002       True  0.672        False  False
vix_pct          


Outcome: next-21-day volatility   (effect = IC for stock rankings, rank correlation for market-state features)
                                             test  effect       t  2010-2015  2016-2021  same sign    p  survives BH   PASS
gk_vol                 ranks stocks within sector   0.523  54.556      0.552      0.492       True  0.0         True   True
downside_vol           ranks stocks within sector   0.446  42.318      0.473      0.419       True  0.0         True   True
vol_of_vol             ranks stocks within sector  -0.012  -1.643      0.000     -0.024      False  0.1        False  False
rel_ret_21             ranks stocks within sector  -0.061  -3.807     -0.051     -0.071       True  0.0         True   True
sue                    ranks stocks within sector  -0.023  -3.574     -0.007     -0.037       True  0.0         True   True
days_since_earnings    ranks stocks within sector   0.085  12.361      0.072      0.098       True  0.0         True   True
vix_pct             

## 9. The same eight on AAPL alone

Part 1 used AAPL from 2020 to 2024. Here AAPL runs from 2010 to 2021 (about 3,000 days), which roughly halves the smallest effect that could be detected.

In [10]:
aapl = pd.DataFrame({k: v[TICKER] for k, v in stock_feats.items()}).join(state).loc[ROUND2[0]:ROUND2[1]]
aapl_tables = {'return': tf.predictive_table(aapl, fwd_ret[TICKER].loc[ROUND2[0]:ROUND2[1]], H),
               'volatility': tf.predictive_table(aapl, fwd_vol[TICKER].loc[ROUND2[0]:ROUND2[1]], H)}
print(pd.concat({k: v[['corr', 't']] for k, v in aapl_tables.items()}, axis=1).round(2).to_string())
print(f"\nAAPL, {len(aapl):,} days: correlations beyond about {2 / np.sqrt(len(aapl) / H):.2f} would stand out at 21 days.")

                    return       volatility      
                      corr     t       corr     t
gk_vol               -0.03 -0.54       0.34  5.67
downside_vol         -0.07 -1.16       0.34  5.85
vol_of_vol            0.03  0.51      -0.14 -1.84
rel_ret_21            0.02  0.33      -0.10 -1.40
sue                  -0.15 -2.29       0.13  2.01
days_since_earnings   0.11  1.91       0.17  3.00
vix_pct               0.13  2.13       0.37  6.50
vix_slope            -0.03 -0.58      -0.27 -4.85

AAPL, 3,021 days: correlations beyond about 0.17 would stand out at 21 days.


## 10. Findings of Part 2

In [11]:
vr, vv = verdicts['return'], verdicts['volatility']
passed_r, passed_v = vr.index[vr['PASS']].tolist(), vv.index[vv['PASS']].tolist()
print(f"RETURN: {len(passed_r)} of 8 features pass ({', '.join(passed_r) or 'none'}).")
top_r = vr['t'].astype(float).abs().idxmax()
print(f"   The strongest is {top_r}: effect {float(vr.loc[top_r, 'effect']):+.3f}, t {float(vr.loc[top_r, 't']):.1f}, same sign in both halves: {bool(vr.loc[top_r, 'same sign'])}.")
print(f"   Earnings surprise (sue): IC {float(vr.loc['sue', 'effect']):+.3f}, t {float(vr.loc['sue', 't']):.1f}; "
      f"halves {float(vr.loc['sue', '2010-2015']):+.3f} and {float(vr.loc['sue', '2016-2021']):+.3f}.")
print(f"VOLATILITY: {len(passed_v)} of 8 features pass ({', '.join(passed_v) or 'none'}).")
top_v = vv['t'].astype(float).abs().idxmax()
print(f"   The strongest is {top_v}: effect {float(vv.loc[top_v, 'effect']):+.3f}, t {float(vv.loc[top_v, 't']):.1f}.")
a_r, a_v = aapl_tables['return'], aapl_tables['volatility']
print(f"AAPL alone: {int((a_r['t'].abs() > 2).sum())} of 8 reach |t| > 2 for its return, {int((a_v['t'].abs() > 2).sum())} of 8 for its volatility.")

J = lambda t: json.loads(t.drop(columns=['test']).astype(float).to_json(orient='index'))
round2 = {'period': ROUND2, 'events': int(len(events)), 'share_dated_by_announcement': float((events['source'] == '8-K').mean()), 'median_lead_days': float(lead.median()),
          'return': J(vr), 'volatility': J(vv), 'tests': vr['test'].to_dict(), 'passed_return': passed_r, 'passed_volatility': passed_v,
          'aapl': {k: json.loads(v.to_json(orient='index')) for k, v in aapl_tables.items()}, 'aapl_days': int(len(aapl))}
with open(os.path.join('data_cache', 'time_series_round2_summary.json'), 'w') as f:
    json.dump(round2, f, indent=1)

RETURN: 2 of 8 features pass (vix_pct, vix_slope).
   The strongest is vix_pct: effect +0.258, t 4.6, same sign in both halves: True.
   Earnings surprise (sue): IC +0.006, t 0.8; halves +0.015 and -0.001.
VOLATILITY: 7 of 8 features pass (gk_vol, downside_vol, rel_ret_21, sue, days_since_earnings, vix_pct, vix_slope).
   The strongest is gk_vol: effect +0.523, t 54.6.
AAPL alone: 2 of 8 reach |t| > 2 for its return, 6 of 8 for its volatility.


## 11. What this does not show

- **No trading.** A correlation is not a profit: turnover and costs come after it.
- **AAPL's sample is one bull market.** March 2020 to August 2024 contains one kind of regime.
- **The S&P 500 test stops in 2021** and still lacks companies that left the index.
- **Many tests were run.** With 21 tests on AAPL and 7 features across the index, a few |t| > 2 results are expected by chance; the sign check across two halves is the guard used here.
- **Only linear, one-feature-at-a-time relationships** are tested. Combinations and interactions are not.
- **The earnings surprise uses no analyst forecasts** (they are not free). It compares a quarter with the same quarter a year earlier, a cruder yardstick than what the market expected.
- **Announcements are used from the next trading day**, so the price jump on the announcement itself is deliberately excluded; only what follows is tested.